<a href="https://colab.research.google.com/github/varshith2589/machine-leanring/blob/main/Flight_Delay_Classification_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df=pd.read_csv("flights.csv")
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.isnull().sum()
df.dropna()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY


In [ ]:
x=df.drop('ARRIVAL_DELAY', axis=1)
y=df['ARRIVAL_DELAY']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()


numeric_cols = X_train.select_dtypes(include=['number']).columns

X_train_numeric = X_train[numeric_cols]
X_test_numeric = X_test[numeric_cols]

X_train_numeric = X_train_numeric.fillna(X_train_numeric.mean())
X_test_numeric = X_test_numeric.fillna(X_test_numeric.mean())


X_train_scaled=scaler.fit_transform(X_train_numeric)
X_test_scaled=scaler.transform(X_test_numeric)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()


numeric_cols = X_train.select_dtypes(include=['number']).columns

X_train_numeric = X_train[numeric_cols]
X_test_numeric = X_test[numeric_cols]

X_train_numeric = X_train_numeric.fillna(X_train_numeric.mean())
X_test_numeric = X_test_numeric.fillna(X_test_numeric.mean())


X_train_scaled=scaler.fit_transform(X_train_numeric)
X_test_scaled=scaler.transform(X_test_numeric)

In [ ]:
from  sklearn.tree import DecisionTreeClassifier
dt_model=DecisionTreeClassifier(max_depth=5,criterion='gini',random_state=42)
dt_model

# Remove NaN values from y_train and y_test, and align X_train_scaled and X_test_scaled
not_nan_mask_train = ~y_train.isna()
X_train_cleaned = X_train_scaled[not_nan_mask_train]
y_train_cleaned = y_train[not_nan_mask_train]

not_nan_mask_test = ~y_test.isna()
X_test_cleaned = X_test_scaled[not_nan_mask_test]
y_test_cleaned = y_test[not_nan_mask_test]

dt_model.fit(X_train_cleaned,y_train_cleaned)
dt_pred=dt_model.predict(X_test_cleaned)
from sklearn.metrics import accuracy_score
dt_accuracy=accuracy_score(y_test_cleaned,dt_pred)
print(dt_accuracy)

0.059056090931554235


In [ ]:
# Convert continuous ARRIVAL_DELAY into a binary classification target
# Define 'delayed' as ARRIVAL_DELAY > 15 minutes, 'not delayed' otherwise
y_train_binary = (y_train_cleaned > 15).astype(int)
y_test_binary = (y_test_cleaned > 15).astype(int)

from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression model
# Set max_iter to a higher value to ensure convergence
log_reg_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=200)

# Train the model using the cleaned and binarized target
log_reg_model.fit(X_train_cleaned, y_train_binary)

# Make predictions on the test set
log_reg_pred = log_reg_model.predict(X_test_cleaned)

# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

log_reg_accuracy = accuracy_score(y_test_binary, log_reg_pred)
print(f"Logistic Regression Accuracy: {log_reg_accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_binary, log_reg_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_binary, log_reg_pred))

Logistic Regression Accuracy: 0.9895

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      5844
           1       0.96      1.00      0.98      2250

    accuracy                           0.99      8094
   macro avg       0.98      0.99      0.99      8094
weighted avg       0.99      0.99      0.99      8094


Confusion Matrix:
[[5759   85]
 [   0 2250]]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(max_depth=3,random_state=42)

# Train the Random Forest model using cleaned and scaled numerical data
rf_model.fit(X_train_cleaned,y_train_cleaned)
rf_pred=rf_model.predict(X_test_cleaned)
from sklearn.metrics import accuracy_score
rf_accuracy=accuracy_score(y_test_cleaned,rf_pred)
print(rf_accuracy)

0.042006424511984185


In [ ]:
model_accuracy = {
    'Decision Tree': dt_accuracy,
    'Logistic Regression': log_reg_accuracy,
    'Random Forest': rf_accuracy
}
model_accuracy

{'Decision Tree': 0.059056090931554235,
 'Logistic Regression': 0.989498393872004,
 'Random Forest': 0.042006424511984185}

In [ ]:
best_model_name=max(model_accuracy,
                   key=model_accuracy.get)
best_accuracy=model_accuracy[best_model_name]
print(f"\n Best Model: {best_model_name} with accuracy of {best_accuracy:.4f}")


 Best Model: Logistic Regression with accuracy of 0.9895


In [ ]:
import joblib
joblib.dump(dt_model,"flights_model.pkl")
joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']

In [ ]:
!pip install sweetviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 59.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import warnings

#Avoid warning's
if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = DeprecationWarning

import sweetviz as sv
import seaborn as sns
#load dataset
df=pd.read_csv("flights.csv")
# generate autamatic EDA Report
report = sv.analyze(df)
report.show_html("flights.csv_AUTOGEN_EDA.html")


from IPython.display import IFrame

# Embed Titanic Sweetviz report
IFrame("flights.csv_AUTOGEN_EDA.html", width=1200, height=600)

                                             |          | [  0%]   00:00 -> (? left)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Report flights.csv_AUTOGEN_EDA.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
